# ENG-209, Partie 2, Série 10

## Classification par Régression Logistique

Dans cet exercice, vous étudierez les données relatives aux ressources humaines déjà présentées en semaine 9.

Cet exercice vise principalement à familiariser les étudiants avec le réglage des différents paramètres et à comprendre comment ceux-ci affectent la qualité et l’ajustement du modèle.

L'objectif est de :

- Expérimenter avec les paramètres du modèle de régression logistique et le seuil de détection afin d'obtenir un taux de fausses négatives inférieur à un maximum fixé, tout en minimisant le taux de fausses positives.

- Évaluer l’effet du surapprentissage (overfitting) sur le modèle.

- Étudier l’impact de la régularisation sur les performances du modèle.

Rappel, dans le context de cet exercice :

- _Fausses négatives_ : le modèle prédit que la personne ne quitte pas l'entreprise, alors qu'elle est partie.

- _Fausses positives_ : le modèle prédit que la personne quitte l'entreprise, alors qu'elle reste.

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.typing import NDArray

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
hr_churn_df=pd.read_csv('../data/hr/hr-churn-1.csv')

In [ ]:
hr_churn_df.info()

### Exploration des données

Voir cours semaine 9.

### Modélisation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import confusion_matrix, classification_report, log_loss, ConfusionMatrixDisplay

#### Split Train/Test

Utilisons 50% des données pour l'apprentissage du model de régression logistique et le reste pour tester le model (test_size=0.5)

⚠️ **Rappel** : Il est crucial de ne pas modifier le paramètre _random_state_ entre les itérations d'optimisation des hyperparamètres. Cela permet d'éviter une contamination des données de test par le processus d'entraînement, ce qui pourrait entraîner un biais dans l'évaluation du modèle. En effet, il est impératif que les données de test ne soient en aucun cas utilisées lors de la sélection des hyperparamètres, tels que le degré ou la régularisation, afin de préserver l'intégrité de la validation.

In [ ]:
X: NDArray[np.float64] =hr_churn_df.loc[:,~hr_churn_df.columns.isin(['Attrition'])].to_numpy()
y: NDArray[np.float64] =hr_churn_df['Attrition'].to_numpy()
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.5, stratify=y, random_state=42)

#### Apprentissage

Essayons de varier les paramètres d'apprentissage du modèle de manière à étudier son comportement.

**Rappel** : Régularisation dans les modèles linéaires/logistiques

**Perte vs Pénalité:**

- **Perte** : mesure l’adéquation du modèle aux données  
  (ex. log-loss pour la régression logistique, erreur quadratique pour la régression linéaire)

- **Pénalité** : contrôle la complexité du modèle, appliquée aux coefficients $w$  

**Pénalités courantes**

- **$\ell_1$ (Lasso)** : $||w||_1$, induit la **sparsité** (on réduit le nombre de coefficients), peut sélectionner des variables, les coefficients des variables non sélectionnées sont à zéro. 

- **$\ell_2$ (Ridge)** : $||w||_2^2$, réduit le surapprentissage, gère la multicolinéarité  

- **Elastic Net** : combinaison convexe de $\ell_1$ et $\ell_2$ (non étudiée dans ce cours)  

- **Pas de pénalité** : utile pour une référence ou une solution de maximum de vraisemblance  

**Force de régularisation**

- **C** dans scikit-learn : inversement proportionnel à la force de régularisation, comme vue pour Lasso et Ridge ($C = 1/\alpha$)  
  - Grand $C$ pour régularisation faible  
  - Petit $C$ pour régularisation forte  

**Solveurs**
- **liblinear** : adapté aux petits jeux de données. Supporte les régularisations $\ell_1$ et $\ell_2$, mais pas Elastic Net. Principalement conçu pour la classification binaire (multiclasse possible avec One-vs-Rest).

- **saga** : supporte les régularisations $\ell_1$, $\ell_2$ et Elastic Net. Solveur stochastique, particulièrement intéressant pour les grands jeux de données et les données creuses (sparse). Nécessite généralement une normalisation des variables pour converger efficacement.

- **lbfgs** : solveur généraliste et solveur par défaut. Bon choix pour la plupart des problèmes de taille petite à moyenne. Supporte uniquement la régularisation $\ell_2$​ ou l'absence de régularisation. Robuste et efficace sur de nombreux types de données.

- **sag** : adapté aux grands jeux de données. Utilise une méthode de gradient stochastique avec moyennage des gradients. Supporte uniquement $\ell_2$​ ou aucune régularisation. Converge efficacement lorsque les variables sont à des échelles comparables.

- **newton-cg** : utilise une méthode de Newton conjuguée pour optimiser la fonction de coût. Supporte uniquement $\ell_2$​ ou aucune régularisation. Peut être efficace sur certains problèmes, notamment lorsque l'on recherche une convergence précise.

Remarque : Ici, nous utilisons des features polynomiales avec la régression logistique. Cela peut sembler surprenant, car la régression logistique est un modèle linéaire, mais en transformant les variables d’entrée avec des polynômes, le modèle devient capable de modéliser des frontières de décision non linéaires tout en restant linéaire par rapport aux coefficients. le but ici est de mesurer le sous/sur apprentissage causé par ce paramètre.

Observez comment le degré influence la performance sur les données d’entraînement et réfléchissez aux conséquences sur la généralisation.

Veuillez répéter les étapes de modélisation et de validation en faisant varier la force de régularisation $C$ et le type de pénalité $\ell_1$, $\ell_2$ avec le solver _liblinear_.

- $C$: plusieur valeurs, entre $0.07$ et $10.0$
- _degree_: 1, 2 et 3

Notez à chaque fois la qualité du modèle, en particulier la fonction de perte logarithmique.

In [ ]:
model: Pipeline =Pipeline(
    [
        ('polynomial', PolynomialFeatures(degree = ???)),
        ('scaler', StandardScaler()),
        ('logistic', LogisticRegression(solver='liblinear', l1_ratio=1, C=???)),
    ])

Voici une fonction qui vous permettra d’évaluer les performances de vos modèles à l’aide des courbes ROC. 

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score, confusion_matrix
from sklearn.base import BaseEstimator

def trace_ROC_PR(model: BaseEstimator,
                X: list[NDArray[np.float64]],
                y: list[NDArray[np.float64]],
                title: str|None=None,
                labels: list[str]|None=None,
                axes: tuple|None=None) -> tuple[Figure, Axes]:
    """
    Affiche la courbe de ROC du model, pour chaque pair de numpy array (X, y).
    
    - model: un modèle de classification
    - X: liste d'ensembles de variables indépendantes X.
    - y: liste d'ensembles de variables cibles y (une pour chaque X)
    - title: titre du graph.
    - label: labels de chaque courbe.
    - axis: le système d'axes retournés par un appel précédant pour superimposer des résultats.

    Retourne le système d'axes.
    """

    # Pour tracer deux graphes, côte à côte (1 ligne et deux colonnes)
    # Ax est un array de dimension lignes x colonnes, chaque ax[i] correspont à un graphe.
    if axes:
        fig, ax = axes
    else:
        fig, ax = plt.subplots(1,2)

    # Pour chaque pair X,y 
    for i in range(0, len(X)):
        label: str = labels[i] if labels else str(i)
        y_predict_proba: NDArray[np.float64] = model.predict_proba(X[i])[:,1]
        
        fpr, tpr, thresholds = roc_curve(y[i], y_predict_proba)
        roc_auc: float = float(roc_auc_score(y_true = y[i], y_score = y_predict_proba))
        
        pre, rec, thresholds = precision_recall_curve(y[i], y_predict_proba)
        avg_pre_rec: float = float(average_precision_score(y_true=y[i], y_score=y_predict_proba))

        ax[0].plot(fpr, tpr, label=f'{label}; AUC: {roc_auc:0.2f}')
        ax[0].plot([0,1],[0,1], color='black', linestyle='--', linewidth=0.5)
        ax[1].plot(rec, pre, label=f'{label}: AUPRC: {avg_pre_rec:0.2f}')

    # ROC
    ax[0].set_title("Courbe de ROC")
    ax[0].set_xlabel("FPR")
    ax[0].set_ylabel("TPR")
    ax[0].legend(loc='best')
    ax[0].grid(True)

    # Précision recall
    ax[1].set_title("Précision-rappel")
    ax[1].set_xlabel("Rappel")
    ax[1].set_ylabel("Précision")
    ax[1].legend(loc='best')
    ax[1].grid(True)

    # Afficher le titre commun à tous les graphs.
    if title:
        fig.suptitle(title)

    # Retourne les axes, afin de pouvoir les réutiliser pour des tracés supplémentaires.
    # plt.show()
    return (fig, ax)

In [ ]:
model.fit(X_train,y_train)

Calculer le nombre de coefficients non nuls (dans le cas de la pénalité Lasso $\ell_1$), par rapport au nombre total de variables indépendantes.

Notez que plus le nombre de variables indépendantes (notamment après transformation polynomiale) est élevé par rapport au nombre de données, plus le modèle risque de surapprendre et de mal généraliser, tandis qu’un nombre trop faible de variables peut entraîner un sous-apprentissage.

In [ ]:
nombre_coef_significatifs = np.sum(np.abs(model.named_steps['logistic'].coef_[0]) > 0.0)
nombre_coef = model.named_steps['logistic'].coef_.shape[1]
print(f"Nombre de coefficients signficatifs: {nombre_coef_significatifs} sur {nombre_coef}")

### Validation du modèle de régression

Nous avons utilisé le _score_ du modèle, qui retourne l’accuracy par défaut. Comme nous le savons, cette métrique peut être trompeuse, notamment pour des jeux de données déséquilibrés, car elle peut rester élevée (donc de bonne qualité) même si le modèle ignore complètement certaines classes.

À la place, nous utilisons le [_log loss_](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html) (ou cross-entropy), qui mesure l'écart entre les vraies classes et les probabilités prédites.
C'est précisément la fonction que la régression logistique minimise, comme vu en cours, et elle constitue un score unique permettant de comparer facilement différents modèles de régression logistique avant d'aborder ultérieurement des métriques à deux dimensions comme la courbe ROC ou la courbe précision-rappel.

Pour rappel:

* Log loss $= −\frac{1}{N} \sum_{​i=1}^N y^i \ln(p^i​)+(1-y^i​) \ln(1−p^i​)$

Où $p^i$ est la probabilité prédite que l'échantillon $i$ soit de classe 1, et donc $(1-p^i)$ qu'il soit de classe 0.

La perte logarithmique est une métrique que l’on cherche à réduire : des valeurs plus faibles indiquent une meilleure adéquation aux données. Attention toutefois au sur-apprentissage, qui peut fausser l’évaluation sur l’ensemble d’entraînement.

#### Mesure de performance du modèle sur les données d'entraînement

On commence par évaluer le modèle sur les données d’entraînement pour vérifier s’il apprend correctement les motifs présents. Une performance faible sur ces données indique un sous-apprentissage, tandis qu’une performance très élevée peut suggérer un surapprentissage, à confirmer ensuite sur les données de test.

In [ ]:
# Premier argument est le vecteur des vraies étiquettes
# Second argument est la table des probabilités prédites pour chaque classe.
log_loss(y_train, model.predict_proba(X_train))

In [ ]:
y_train_pred: NDArray[np.float64] = np.asarray(model.predict(X_train))

In [ ]:
plt.rcParams["figure.figsize"] = (6, 4)

cm: NDArray[np.float64] = confusion_matrix(y_true=???, y_pred=???)
disp: ConfusionMatrixDisplay = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=???)

disp.plot()
plt.show()

In [ ]:
print(f'{classification_report(y_true=???,y_pred=???, target_names=???)}')

In [ ]:
_ = trace_ROC_PR(model, [???], [???], labels=["RL Train"])

#### Mesure de performance du modèle sur les données de test

In [ ]:
log_loss(???,???)

In [ ]:
y_pred: NDArray[np.float64] = np.asarray(???)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 4)

cm = confusion_matrix(y_true=???, y_pred=???)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=???)

disp.plot()
plt.show()

In [ ]:
print(f'{classification_report(y_true=???,y_pred=???, target_names=???)}')

In [ ]:
_ = trace_ROC_PR(model, [???, ???], [???, ???], labels=["RL Train", "RL Test"])

Notez les variations de performance pour les différentes valeurs de $C$ et de degré. Que pouvez‑vous en conclure ?

- Effet du degré : ...

- Effet de $C$ : ...

### Explorons maintenant l’effet du seuil de détection

#### Choix du seuil de détection

Choisissez la valeur de régularisation et le degré qui donnent le meilleur modèle.

Détermination manuelle du seuil de détection : L'objectif est de définir un seuil de détection à partir des données d'entraînement de manière à ce que le taux de faux positifs soit limité à un maximum de 10 % des positifs réels, tout en minimisant le taux de faux négatifs. Les faux positifs correspondent aux personnes qui, malgré une prédiction de départ, restent effectivement.

In [ ]:
threshold = ???
probs = model.predict_proba(X_train)
y_train_pred_th = (probs[:, 1] >= threshold).astype(int)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 4)

cm = confusion_matrix(y_true=???, y_pred=???)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=???)

disp.plot()
plt.show()

In [ ]:
print(f'{classification_report(y_true=???,y_pred=???, target_names=???)}')

#### Validation du seuil de détection

Veuillez maintenant vérifier l'effet du seuil de détection sur les données de test.

In [ ]:
probs = model.predict_proba(X_test)
y_test_pred_th: NDArray[np.float64] = (probs[:, 1] >= threshold).astype(int)

In [ ]:
plt.rcParams["figure.figsize"] = (6, 4)

cm = confusion_matrix(y_true=???, y_pred=???)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=???)

disp.plot()
plt.show()

In [ ]:
print(f'{classification_report(y_true=???,y_pred=???, target_names=???)}')